
# GuardianX – DeBERTa High-Accuracy Notebook (NYC 311)

This notebook supports:
- Full NYC 311 CSV loading
- Long sentence intent recognition
- Combined Complaint Type + Descriptor
- Auto-balanced classes
- DeBERTa model with high accuracy


In [ ]:
!pip install -U transformers datasets accelerate


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import pandas as pd
import numpy as np
import re
import torch
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments


In [ ]:

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)   # remove symbols
    text = re.sub(r"\s+", " ", text).strip()     # clean spaces
    return text



In [ ]:

KEYWORD_TO_CLASS = {
    "fire": "Emergency",
    "accident": "Emergency",
    "emergency": "Emergency",
    "ambulance": "Emergency",

    "traffic": "Traffic",
    "jam": "Traffic",
    "signal": "Traffic",
    "road": "Traffic",

    "garbage": "Garbage",
    "waste": "Garbage",
    "trash": "Garbage",
    "dustbin": "Garbage",

    "water": "Water",
    "leak": "Water",
    "pipe": "Water",
    "tap": "Water",
    "supply": "Water",

    "electricity": "Electricity",
    "electric": "Electricity",
    "power": "Electricity",
    "light": "Electricity"
}

def assign_label(text):
    text = preprocess_text(text)
    for k, v in KEYWORD_TO_CLASS.items():
        if k in text:
            return v
    return None


In [ ]:

CSV_PATH = "/content/drive/MyDrive/311_Service_Requests_from_2011.csv"
df = pd.read_csv(CSV_PATH, low_memory=False)


In [ ]:
import unicodedata

def normalize_col(c):
    c = unicodedata.normalize("NFKD", c)   # remove unicode weirdness
    c = c.encode("ascii", "ignore").decode("ascii")
    c = c.strip().lower()
    c = " ".join(c.split())
    return c

df.columns = [normalize_col(c) for c in df.columns]

print("Normalized columns:")
for i, c in enumerate(df.columns):
    print(i, repr(c))


In [ ]:
def find_col(keyword_list):
    for col in df.columns:
        for kw in keyword_list:
            if kw in col:
                return col
    return None

agency_col = find_col(["agency"])
complaint_col = find_col(["complaint"])
descriptor_col = find_col(["descriptor", "description", "desc"])

print("Detected columns:")
print("Agency:", agency_col)
print("Complaint:", complaint_col)
print("Descriptor:", descriptor_col)

text_cols = [c for c in [agency_col, complaint_col, descriptor_col] if c is not None]

if not text_cols:
    raise ValueError("No valid text columns found even after auto-detection")

df["text"] = df[text_cols].astype(str).agg(" ".join, axis=1)
df = df[["text"]].dropna()

print("Using columns:", text_cols)
print("After combine:", df.shape)


In [ ]:

df["label"] = df["text"].apply(assign_label)
df = df.dropna(subset=["label"])
df["text"] = df["text"].apply(preprocess_text)
df = df[["text", "label"]]
print(df["label"].value_counts())


In [ ]:

min_size = df["label"].value_counts().min()
balanced = []

for label in df["label"].unique():
    subset = df[df["label"] == label]
    balanced.append(resample(subset, n_samples=min_size, random_state=42))

df = pd.concat(balanced).sample(frac=1).reset_index(drop=True)


In [ ]:

label_encoder = LabelEncoder()
df["labels"] = label_encoder.fit_transform(df["label"])

num_labels = len(label_encoder.classes_)


In [ ]:

MODEL_NAME = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)


In [ ]:

MAX_LEN = 128

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )


In [ ]:

from sklearn.model_selection import train_test_split
from datasets import Dataset

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize, batched=True)

train_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)


In [ ]:
training_args = TrainingArguments(
    output_dir="./guardianx_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs"
)

In [ ]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer
)

trainer.train()


In [ ]:

def predict_department(text):
    text = preprocess_text(text)
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    confidence, pred = torch.max(probs, dim=1)
    label = label_encoder.inverse_transform(pred.cpu().numpy())[0]

    if confidence.item() < 0.55:
        return "General Support", confidence.item()

    return label, confidence.item()


In [ ]:
MODEL_SAVE_PATH = "/content/drive/MyDrive/GuardianX_DeBERTa_Final"


In [ ]:
# Save model & tokenizer
trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

print("Model and tokenizer saved to:", MODEL_SAVE_PATH)


In [ ]:
import joblib

joblib.dump(label_encoder, f"{MODEL_SAVE_PATH}/label_encoder.pkl")

print("Label encoder saved")


In [ ]:
import os

os.listdir(MODEL_SAVE_PATH)


*start* from here

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import joblib
import torch
import torch.nn.functional as F

MODEL_SAVE_PATH = "/content/drive/MyDrive/GuardianX_DeBERTa_Final"

tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_SAVE_PATH)
label_encoder = joblib.load(f"{MODEL_SAVE_PATH}/label_encoder.pkl")

model.eval()
print("Model loaded successfully")


In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import login, upload_folder
import os

# SECURITY: never hardcode tokens in the notebook.
# In Colab: Settings (key icon in left sidebar) -> Secrets -> add HF_TOKEN, then:
#   from google.colab import userdata
#   hf_token = userdata.get("HF_TOKEN")
# Locally / elsewhere: set an environment variable before launching Jupyter, e.g.
#   export HF_TOKEN="hf_xxx...";  jupyter notebook
# Fallback below prompts securely at runtime if no env var / secret is found.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    import getpass
    hf_token = getpass.getpass("Enter your Hugging Face token (input hidden): ")

login(token=hf_token)

MODEL_SAVE_PATH = "/content/drive/MyDrive/GuardianX_DeBERTa_Final"  # your existing path

upload_folder(
    folder_path=MODEL_SAVE_PATH,
    repo_id="Ayushgupta301/guardianx",   # your repo id from A3
    repo_type="model",
)


In [ ]:
def predict_with_fallback(text, threshold=0.55):
    text = preprocess_text(text)

    # ✅ First: keyword match
    for k, v in KEYWORD_TO_CLASS.items():
        if k in text:
            return v, 1.0

    # Then use model
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    confidence, pred = torch.max(probs, dim=1)

    label = label_encoder.inverse_transform(pred.cpu().numpy())[0]

    return label, confidence.item()



In [ ]:
# ============================================
# REQUIRED DEFINITIONS FOR GRADIO INTERFACE
# ============================================

import re

# Text preprocessing
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Keyword mapping
KEYWORD_TO_CLASS = {
    # Emergency
    "fire": "Emergency",
    "accident": "Emergency",
    "ambulance": "Emergency",

    # Traffic
    "traffic": "Traffic",
    "jam": "Traffic",
    "signal": "Traffic",
    "road": "Traffic",

    # Garbage
    "garbage": "Garbage",
    "waste": "Garbage",
    "trash": "Garbage",
    "dustbin": "Garbage",

    # Water
    "water": "Water",
    "leak": "Water",
    "pipe": "Water",
    "tap": "Water",
    "supply": "Water",

    # Electricity  ✅
    "electric": "Electricity",
    "electricity": "Electricity",
    "power": "Electricity",
    "light": "Electricity",
    "current": "Electricity",
    "voltage": "Electricity"
}


# SLA Table (Service Level Agreement times)
SLA_TABLE = {
    "Emergency": "00:30",    # 30 minutes
    "Water": "02:00",        # 2 hours
    "Electricity": "04:00",  # 4 hours
    "Traffic": "01:00",      # 1 hour
    "Garbage": "24:00"       # 24 hours
}

print("✓ All required definitions loaded")

In [ ]:
!pip install -q gradio joblib


In [ ]:
# ==================== CELL 3: Import All Libraries ====================
import pandas as pd
import numpy as np
import re
import torch
import torch.nn.functional as F
import gradio as gr
import sqlite3
import threading
import time
import datetime
import random
import smtplib
import requests
import matplotlib.pyplot as plt
from email.mime.text import MIMEText
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import joblib

print("✅ All libraries imported successfully")

In [ ]:
# ==================== CELL 4: Load Trained Model ====================
MODEL_SAVE_PATH = "/content/drive/MyDrive/GuardianX_DeBERTa_Final"

tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_SAVE_PATH)
label_encoder = joblib.load(f"{MODEL_SAVE_PATH}/label_encoder.pkl")

model.eval()
print("Model loaded successfully")


In [ ]:
# ==================== CELL 5: Configuration ====================
KEYWORD_TO_CLASS = {
    "fire": "Emergency", "accident": "Emergency", "emergency": "Emergency", "ambulance": "Emergency",
    "traffic": "Traffic", "jam": "Traffic", "signal": "Traffic", "road": "Traffic",
    "garbage": "Garbage", "waste": "Garbage", "trash": "Garbage", "dustbin": "Garbage",
    "water": "Water", "leak": "Water", "pipe": "Water", "tap": "Water", "supply": "Water",
    "electricity": "Electricity", "electric": "Electricity", "power": "Electricity", "light": "Electricity"
}

ACTIONS = {
    "Emergency": ["Fire Agent", "Medical Agent"],
    "Traffic": ["Traffic Control", "Road Safety"],
    "Garbage": ["Sanitation Team", "Waste Management"],
    "Water": ["Water Supply", "Plumbing Team"],
    "Electricity": ["Power Grid", "Line Maintenance"]
}

AGENT_DECISIONS = {
    "Emergency": ["Alert control room", "Dispatch fire unit", "Dispatch medical unit", "Resolve"],
    "Traffic": ["Assess traffic flow", "Deploy traffic officers", "Update signals", "Resolve"],
    "Garbage": ["Schedule pickup", "Dispatch sanitation team", "Clear area", "Resolve"],
    "Water": ["Check water supply", "Dispatch repair team", "Fix issue", "Resolve"],
    "Electricity": ["Locate outage", "Dispatch technicians", "Restore power", "Resolve"]
}

SLA_TABLE = {
    "Emergency": "00:30", "Traffic": "02:00", "Garbage": "24:00",
    "Water": "12:00", "Electricity": "08:00"
}

# ==================== SECRETS (do NOT hardcode) ====================
# SECURITY: credentials are loaded from environment variables / Colab Secrets,
# never written into the notebook. Set these before running:
#
#   Colab:  Settings (key icon) -> Secrets -> add EMAIL_FROM, EMAIL_APP_PASSWORD
#   Local:  export EMAIL_FROM="you@gmail.com"
#           export EMAIL_APP_PASSWORD="xxxxxxxxxxxxxxxx"   # Gmail App Password, not your login password
#
# EMAIL_APP_PASSWORD must be a Gmail "App Password" (Google Account -> Security ->
# 2-Step Verification -> App passwords), not your real Gmail account password.
import os

try:
    from google.colab import userdata
    EMAIL_FROM = userdata.get("EMAIL_FROM")
    EMAIL_PASSWORD = userdata.get("EMAIL_APP_PASSWORD")
except Exception:
    EMAIL_FROM = os.environ.get("EMAIL_FROM")
    EMAIL_PASSWORD = os.environ.get("EMAIL_APP_PASSWORD")

if not EMAIL_FROM or not EMAIL_PASSWORD:
    print("⚠️ EMAIL_FROM / EMAIL_APP_PASSWORD not set — email notifications will be disabled.")
    print("   Set them as environment variables or Colab Secrets to enable ticket emails.")

API_URL = os.environ.get("GUARDIANX_WEBHOOK_URL", "https://webhook.site/your-unique-url")
CREATOR_NAME = ""

print("Configuration loaded")


In [ ]:
# ==================== CELL 6: Database Setup ====================
def view_db():
    try:
        conn = sqlite3.connect("complaints.db")
        cur = conn.cursor()
        cur.execute("""
            SELECT ticket_no, created_by, department, agent, complaint,
                   location, priority, status, created_at, closed_at
            FROM tickets ORDER BY created_at DESC
        """)
        rows = cur.fetchall()
        conn.close()

        if len(rows) == 0:
            return pd.DataFrame(columns=[
                "Ticket No", "Created By", "Department", "Agent",
                "Complaint", "Location", "Priority", "Status",
                "Created At", "Closed At"
            ])

        df = pd.DataFrame(rows, columns=[
            "Ticket No", "Created By", "Department", "Agent",
            "Complaint", "Location", "Priority", "Status",
            "Created At", "Closed At"
        ])
        df["Complaint"] = df["Complaint"].apply(lambda x: x[:40] + "..." if len(str(x)) > 40 else x)
        return df
    except Exception as e:
        print(f"❌ Database view error: {e}")
        return pd.DataFrame()

def get_ticket_stats():
    try:
        conn = sqlite3.connect("complaints.db")
        cur = conn.cursor()

        cur.execute("SELECT COUNT(*) FROM tickets")
        total = cur.fetchone()[0]

        cur.execute("SELECT COUNT(*) FROM tickets WHERE status='OPEN'")
        open_count = cur.fetchone()[0]

        cur.execute("SELECT COUNT(*) FROM tickets WHERE status='CLOSED'")
        closed_count = cur.fetchone()[0]

        conn.close()

        return f"📊 Total: {total} | 🟢 Open: {open_count} | ✅ Closed: {closed_count}"
    except Exception as e:
        return f"Stats unavailable: {str(e)}"



In [ ]:
# ==================== CELL 7: Utility Functions ====================
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def predict_with_fallback(text, threshold=0.55):
    text = preprocess_text(text)
    for k, v in KEYWORD_TO_CLASS.items():
        if k in text:
            return v, 1.0

    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                      padding="max_length", max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    confidence, pred = torch.max(probs, dim=1)

    if confidence.item() < threshold:
        return "General Support", confidence.item()

    label = label_encoder.inverse_transform(pred.cpu().numpy())[0]
    return label, confidence.item()

def detect_priority(text):
    text = text.lower()
    if any(word in text for word in ["emergency", "urgent", "fire", "accident", "critical"]):
        return "🔴 Critical"
    elif any(word in text for word in ["important", "soon", "asap"]):
        return "🟠 High"
    elif any(word in text for word in ["moderate", "medium"]):
        return "🟡 Medium"
    return "🟢 Low"

def send_email(subject, body, recipient_email):
    """Modified to accept recipient email as parameter"""
    if not EMAIL_FROM or not EMAIL_PASSWORD:
        print("⚠️ Email not configured (EMAIL_FROM / EMAIL_APP_PASSWORD missing) — skipping send.")
        return False
    try:
        msg = MIMEText(body)
        msg["Subject"] = subject
        msg["From"] = EMAIL_FROM
        msg["To"] = recipient_email  # Use the provided email

        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(EMAIL_FROM, EMAIL_PASSWORD)
            server.send_message(msg)
        print(f"📧 Email sent to: {recipient_email}")
        return True
    except Exception as e:
        print(f"❌ Email error: {e}")
        return False

def call_api(agent, text, location):
    try:
        payload = {
            "agent": agent,
            "complaint": text,
            "location": location,
            "timestamp": datetime.datetime.now().isoformat()
        }
        response = requests.post(API_URL, json=payload, timeout=5)
        print(f"🌐 API called: {response.status_code}")
    except Exception as e:
        print(f"❌ API error: {e}")

In [ ]:
# ==================== CELL 8: Global State ====================
data_store = []
history = []
ticket_status = {}

def add_to_history(message):
    timestamp = datetime.datetime.now().strftime("%H:%M:%S")
    history.append(f"[{timestamp}] {message}")
    if len(history) > 25:
        history.pop(0)

def get_history_text():
    return "\n".join(history)

def get_ticket_followup(ticket_no):
    return ticket_status.get(ticket_no, "No active ticket")

In [ ]:
def agent_job(agent, text, location, priority, creator_name, user_email):
    """Create ticket and save to database - NO AUTO-CLOSE"""
    ticket_no = f"TKT-{random.randint(100000, 999999)}"
    department = agent

    try:
        add_to_history("")
        add_to_history("="*60)
        add_to_history(f"🎫 CREATING TICKET IN DATABASE")
        add_to_history("="*60)
        add_to_history(f"🎫 Ticket Number: {ticket_no}")
        add_to_history(f"👤 Created by: {creator_name}")
        add_to_history(f"📧 Email: {user_email}")
        add_to_history(f"🏢 Department: {department}")
        add_to_history(f"🤖 Agent: {agent}")
        add_to_history(f"📍 Location: {location}")
        add_to_history(f"⚠️ Priority: {priority}")
        add_to_history(f"📊 Status: OPEN")

        # Save to database
        conn = sqlite3.connect("complaints.db")
        cur = conn.cursor()
        created_time = datetime.datetime.now().isoformat()

        cur.execute("""
            INSERT INTO tickets
            (ticket_no, created_by, email, department, agent, complaint, location, priority, status, created_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, 'OPEN', ?)
        """, (ticket_no, creator_name, user_email, department, agent, text, location, priority, created_time))

        conn.commit()
        conn.close()

        add_to_history(f"✅ Ticket {ticket_no} saved to database successfully!")
        add_to_history(f"📊 Database status: OPEN")

        # Update data store for analytics
        data_store.append({
            "ticket_no": ticket_no,
            "agent": agent,
            "priority": priority,
            "location": location,
            "date": datetime.date.today().isoformat(),
            "status": "OPEN"
        })

        # Send opening email
        add_to_history(f"📧 Sending opening email to: {user_email}")
        email_sent = send_email(
            f"🆕 New {department} Complaint - {ticket_no}",
            f"""NEW TICKET CREATED

Ticket Number: {ticket_no}
Created By: {creator_name}
Department: {department}
Agent: {agent}
Complaint: {text}
Location: {location}
Priority: {priority}
Status: OPEN
Created At: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Your ticket has been assigned to the {department} Department.
You will receive another email when this ticket is resolved.

Please keep this ticket number for your reference: {ticket_no}

Thank you for using GuardianX AI!
""",
            user_email
        )

        if email_sent:
            add_to_history(f"✅ Opening email sent successfully to {user_email}")
        else:
            add_to_history(f"⚠️ Email sending failed (check email configuration)")

        # Call external API
        call_api(agent, text, location)
        add_to_history(f"🌐 External API notified")

        add_to_history(f"📌 Ticket {ticket_no} is now OPEN - awaiting {department} Department resolution")
        add_to_history("="*60)

        # Update ticket status in memory
        ticket_status[ticket_no] = f"⏳ {agent} Department - Ticket OPEN, awaiting resolution"

        return ticket_no

    except Exception as e:
        print(f"❌ Agent job error: {e}")
        import traceback
        traceback.print_exc()
        add_to_history(f"❌ Error creating ticket: {str(e)}")
        return None

def run_real_agent(agent, text, location, priority, creator_name, user_email):
    """Run agent in thread"""
    ticket_container = [None]

    def thread_wrapper():
        ticket_no = agent_job(agent, text, location, priority, creator_name, user_email)
        ticket_container[0] = ticket_no

    thread = threading.Thread(target=thread_wrapper, daemon=True)
    thread.start()
    thread.join(timeout=3)  # Wait up to 3 seconds for completion

    return ticket_container[0]

In [ ]:
# ==================== CELL 11: Main Run Agent ====================
current_ticket = None

def run_agent(text, location, creator_name=None, user_email=None):
    """Main function that processes complaints and creates tickets"""
    global current_ticket

    try:
        if not text or not location:
            return "", "", "", "", "", get_history_text()

        if not creator_name or creator_name.strip() == "":
            creator_name = "Anonymous"

        # Validate email
        if not user_email or user_email.strip() == "":
            add_to_history("❌ Error: Email is required")
            return "Error: Email is required", "", "", "", "", get_history_text()

        # Get AI prediction for department
        agent, confidence = predict_with_fallback(text)
        agent = agent.title()
        priority = detect_priority(text)

        sub_agents = ", ".join(ACTIONS.get(agent, ["General Support"]))
        decisions = AGENT_DECISIONS.get(agent, ["Analyze", "Assign", "Resolve"])
        decision_text = "\n".join([f"{i+1}. {d}" for i, d in enumerate(decisions)])
        sla = SLA_TABLE.get(agent, "24:00")

        add_to_history("")
        add_to_history("="*60)
        add_to_history(f"🔍 NEW COMPLAINT ANALYSIS STARTED")
        add_to_history("="*60)
        add_to_history(f"📝 Complaint: '{text[:50]}{'...' if len(text) > 50 else ''}'")
        add_to_history(f"📍 Location: {location}")
        add_to_history(f"👤 Submitted by: {creator_name}")
        add_to_history(f"📧 Email: {user_email}")
        add_to_history(f"🤖 AI Analysis: Category identified as {agent} (confidence: {confidence:.2%})")
        add_to_history(f"⏱️ SLA: {sla}")
        add_to_history("")

        # Create ticket through agent system
        ticket_no = run_real_agent(agent, text, location, priority, creator_name, user_email)
        current_ticket = ticket_no

        if ticket_no:
            add_to_history(f"✅ Ticket {ticket_no} successfully created and saved")
        else:
            add_to_history("❌ Error creating ticket")

        return agent, sub_agents, sla, priority, decision_text, get_history_text()

    except Exception as e:
        print(f"🔥 RUN_AGENT ERROR: {e}")
        import traceback
        traceback.print_exc()
        error_msg = f"Error: {str(e)}"
        add_to_history(f"❌ Critical Error: {str(e)}")
        return error_msg, error_msg, error_msg, error_msg, error_msg, get_history_text()

In [ ]:
# ==================== CELL 12: Analytics Functions ====================
def generate_charts():
    if not data_store:
        return None, None, None

    try:
        df = pd.DataFrame(data_store)

        plt.figure(figsize=(8, 6))
        df["agent"].value_counts().plot(kind="bar", color='orange')
        plt.title("Complaints by Agent", fontsize=14, fontweight='bold')
        plt.xlabel("Agent")
        plt.ylabel("Count")
        plt.tight_layout()
        plt.savefig("agent.png")
        plt.clf()

        plt.figure(figsize=(8, 6))
        df["date"].value_counts().sort_index().plot(kind="line", marker='o', color='orange', linewidth=2)
        plt.title("Complaints by Day", fontsize=14, fontweight='bold')
        plt.xlabel("Date")
        plt.ylabel("Count")
        plt.tight_layout()
        plt.savefig("day.png")
        plt.clf()

        plt.figure(figsize=(8, 6))
        df["priority"].value_counts().plot(kind="pie", autopct="%1.1f%%",
                                          colors=['red', 'orange', 'yellow', 'green'],
                                          startangle=90)
        plt.title("Complaints by Priority", fontsize=14, fontweight='bold')
        plt.ylabel("")
        plt.tight_layout()
        plt.savefig("priority.png")
        plt.clf()

        return "agent.png", "day.png", "priority.png"
    except Exception as e:
        print(f"❌ Chart generation error: {e}")
        return None, None, None

def get_ticket_stats():
    try:
        conn = sqlite3.connect("complaints.db")
        cur = conn.cursor()

        cur.execute("SELECT COUNT(*) FROM tickets")
        total = cur.fetchone()[0]

        cur.execute("SELECT COUNT(*) FROM tickets WHERE status='OPEN'")
        open_count = cur.fetchone()[0]

        cur.execute("SELECT COUNT(*) FROM tickets WHERE status='CLOSED'")
        closed_count = cur.fetchone()[0]

        conn.close()

        return f"📊 Total: {total} | 🟢 Open: {open_count} | ✅ Closed: {closed_count}"
    except Exception as e:
        return f"Stats unavailable: {str(e)}"

def export_admin():
    try:
        conn = sqlite3.connect("complaints.db")
        df = pd.read_sql_query("SELECT * FROM tickets ORDER BY created_at DESC", conn)
        conn.close()

        if len(df) == 0:
            print("⚠️ No tickets to export")
            return None

        filename = f"guardianx_tickets_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(filename, index=False)
        print(f"✅ Exported {len(df)} tickets to {filename}")
        return filename
    except Exception as e:
        print(f"❌ Export error: {e}")
        return None


In [ ]:
# ==================== DATABASE INITIALIZATION ====================
def init_db():
    """Initialize database with email column"""
    conn = sqlite3.connect("complaints.db")
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS tickets")
    cur.execute("""
        CREATE TABLE tickets (
            ticket_no TEXT PRIMARY KEY,
            created_by TEXT NOT NULL,
            email TEXT NOT NULL,
            department TEXT NOT NULL,
            agent TEXT NOT NULL,
            complaint TEXT NOT NULL,
            location TEXT NOT NULL,
            priority TEXT,
            status TEXT DEFAULT 'OPEN',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            closed_at TIMESTAMP,
            resolution TEXT,
            resolved_by TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Database initialized with email column")

In [ ]:
# ==================== COMPLETE GUARDIANX GRADIO INTERFACE ====================
# Add this to your existing code after all the function definitions

import gradio as gr
import pandas as pd
import sqlite3
import datetime
import html as _html


# ==================== SMALL DISPLAY HELPERS ====================
def _esc(value):
    """HTML-escape any value, tolerating None."""
    return _html.escape(str(value)) if value is not None else ""

def _fmt_dt(value):
    """Turn an ISO timestamp into a friendly 'DD Mon YYYY, HH:MM' string."""
    if not value:
        return "—"
    text = str(value)
    for fmt in ("%Y-%m-%dT%H:%M:%S.%f", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S.%f", "%Y-%m-%d %H:%M:%S"):
        try:
            return datetime.datetime.strptime(text, fmt).strftime("%d %b %Y, %I:%M %p")
        except ValueError:
            continue
    return text.replace("T", " ")[:16]

def _priority_badge(priority):
    p = str(priority or "")
    if "Critical" in p:
        cls = "badge-critical"
    elif "High" in p:
        cls = "badge-high"
    elif "Medium" in p:
        cls = "badge-medium"
    else:
        cls = "badge-low"
    label = p.split(" ", 1)[-1] if " " in p else (p or "Low")
    return f'<span class="badge {cls}">{_esc(label)}</span>'

def _status_badge(status):
    s = str(status or "").upper()
    cls = "badge-status-open" if s == "OPEN" else "badge-status-closed"
    icon = "●" if s == "OPEN" else "✓"
    return f'<span class="badge {cls}">{icon} {_esc(s or "UNKNOWN")}</span>'

def _empty_state(icon, title, subtitle):
    return f"""
    <div class="gx-empty-state">
        <div class="gx-empty-icon">{icon}</div>
        <div class="gx-empty-title">{_esc(title)}</div>
        <div class="gx-empty-sub">{_esc(subtitle)}</div>
    </div>
    """

def _stat_row(total, open_count, closed_count):
    return f"""
    <div class="gx-stat-row">
        <div class="gx-stat-card">
            <div class="gx-stat-icon gx-stat-total">📊</div>
            <div><div class="gx-stat-num">{total}</div><div class="gx-stat-label">Total Tickets</div></div>
        </div>
        <div class="gx-stat-card">
            <div class="gx-stat-icon gx-stat-open">●</div>
            <div><div class="gx-stat-num">{open_count}</div><div class="gx-stat-label">Open</div></div>
        </div>
        <div class="gx-stat-card">
            <div class="gx-stat-icon gx-stat-closed">✓</div>
            <div><div class="gx-stat-num">{closed_count}</div><div class="gx-stat-label">Closed</div></div>
        </div>
    </div>
    """


# ==================== DEPARTMENT FUNCTIONS ====================
def get_department_tickets_html(department):
    """Build a card list of OPEN tickets for a specific department."""
    try:
        conn = sqlite3.connect("complaints.db")
        cur = conn.cursor()
        cur.execute("""
            SELECT ticket_no, created_by, complaint, location,
                   priority, status, created_at
            FROM tickets
            WHERE department = ? AND status = 'OPEN'
            ORDER BY created_at DESC
        """, (department,))
        rows = cur.fetchall()
        conn.close()

        print(f"📊 Found {len(rows)} OPEN tickets for {department} department")

        if not rows:
            return _empty_state("📭", "No open tickets", f"{department} department has no pending tickets right now.")

        cards = []
        for ticket_no, created_by, complaint, location, priority, status, created_at in rows:
            cards.append(f"""
            <div class="ticket-card">
                <div class="ticket-card-head">
                    <span class="ticket-id">{_esc(ticket_no)}</span>
                    <div class="ticket-badges">{_priority_badge(priority)}{_status_badge(status)}</div>
                </div>
                <div class="ticket-meta">
                    <div class="ticket-meta-item"><span class="mi-label">Citizen</span><span class="mi-value">{_esc(created_by)}</span></div>
                    <div class="ticket-meta-item"><span class="mi-label">Location</span><span class="mi-value">{_esc(location)}</span></div>
                    <div class="ticket-meta-item"><span class="mi-label">Submitted</span><span class="mi-value">{_esc(_fmt_dt(created_at))}</span></div>
                </div>
                <div class="ticket-complaint">{_esc(complaint)}</div>
            </div>
            """)
        return f'<div class="ticket-grid">{"".join(cards)}</div>'
    except Exception as e:
        print(f"❌ Error fetching department tickets: {e}")
        import traceback
        traceback.print_exc()
        return _empty_state("⚠️", "Couldn't load tickets", str(e))

def get_department_closed_tickets_html(department):
    """Build a card list of CLOSED tickets for a specific department."""
    try:
        conn = sqlite3.connect("complaints.db")
        cur = conn.cursor()
        cur.execute("""
            SELECT ticket_no, created_by, complaint, location,
                   priority, status, created_at, resolution, closed_at, resolved_by
            FROM tickets
            WHERE department = ? AND status = 'CLOSED'
            ORDER BY closed_at DESC
        """, (department,))
        rows = cur.fetchall()
        conn.close()

        print(f"📊 Found {len(rows)} CLOSED tickets for {department} department")

        if not rows:
            return _empty_state("📭", "No closed tickets yet", f"Resolved {department} tickets will appear here.")

        cards = []
        for ticket_no, created_by, complaint, location, priority, status, created_at, resolution, closed_at, resolved_by in rows:
            cards.append(f"""
            <div class="ticket-card ticket-card-closed">
                <div class="ticket-card-head">
                    <span class="ticket-id">{_esc(ticket_no)}</span>
                    <div class="ticket-badges">{_priority_badge(priority)}{_status_badge(status)}</div>
                </div>
                <div class="ticket-meta">
                    <div class="ticket-meta-item"><span class="mi-label">Citizen</span><span class="mi-value">{_esc(created_by)}</span></div>
                    <div class="ticket-meta-item"><span class="mi-label">Location</span><span class="mi-value">{_esc(location)}</span></div>
                    <div class="ticket-meta-item"><span class="mi-label">Submitted</span><span class="mi-value">{_esc(_fmt_dt(created_at))}</span></div>
                    <div class="ticket-meta-item"><span class="mi-label">Closed</span><span class="mi-value">{_esc(_fmt_dt(closed_at))}</span></div>
                </div>
                <div class="ticket-complaint">{_esc(complaint)}</div>
                <div class="ticket-resolution"><span class="mi-label">Resolution</span> {_esc(resolution)}</div>
            </div>
            """)
        return f'<div class="ticket-grid">{"".join(cards)}</div>'
    except Exception as e:
        print(f"❌ Error fetching closed tickets: {e}")
        import traceback
        traceback.print_exc()
        return _empty_state("⚠️", "Couldn't load tickets", str(e))

def get_department_stats(department):
    """Get statistics HTML for a specific department"""
    try:
        conn = sqlite3.connect("complaints.db")
        cur = conn.cursor()

        cur.execute("SELECT COUNT(*) FROM tickets WHERE department = ?", (department,))
        total = cur.fetchone()[0]

        cur.execute("SELECT COUNT(*) FROM tickets WHERE department = ? AND status = 'OPEN'", (department,))
        open_count = cur.fetchone()[0]

        cur.execute("SELECT COUNT(*) FROM tickets WHERE department = ? AND status = 'CLOSED'", (department,))
        closed_count = cur.fetchone()[0]

        conn.close()

        print(f"📊 {department} stats - Total: {total}, Open: {open_count}, Closed: {closed_count}")

        return _stat_row(total, open_count, closed_count)
    except Exception as e:
        print(f"❌ Error getting stats: {e}")
        return _empty_state("⚠️", "Stats unavailable", str(e))

def mark_ticket_resolved(ticket_no, resolution_note, department):
    """Mark ticket as resolved - ONLY way to close tickets"""
    try:
        if not ticket_no or not ticket_no.strip():
            return "❌ Error: Please enter a ticket number", get_history_text()

        if not resolution_note or not resolution_note.strip():
            return "❌ Error: Please enter resolution details", get_history_text()

        conn = sqlite3.connect("complaints.db")
        cur = conn.cursor()

        # Get ticket details
        cur.execute("""
            SELECT created_by, complaint, location, department, email
            FROM tickets
            WHERE ticket_no = ? AND status = 'OPEN'
        """, (ticket_no,))
        ticket_data = cur.fetchone()

        if not ticket_data:
            conn.close()
            return f"❌ Error: Ticket {ticket_no} not found or already closed", get_history_text()

        created_by, complaint, location, ticket_dept, email = ticket_data

        # Check department match
        if ticket_dept != department:
            conn.close()
            return f"❌ Error: This ticket belongs to {ticket_dept} department, not {department}", get_history_text()

        closed_time = datetime.datetime.now().isoformat()
        resolved_by_text = f"{department} Department"

        # Update ticket
        cur.execute("""
            UPDATE tickets
            SET status = 'CLOSED',
                resolution = ?,
                closed_at = ?,
                resolved_by = ?
            WHERE ticket_no = ?
        """, (resolution_note, closed_time, resolved_by_text, ticket_no))

        conn.commit()
        conn.close()

        # Add to history
        add_to_history("")
        add_to_history("="*60)
        add_to_history(f"✅ TICKET MANUALLY RESOLVED BY DEPARTMENT")
        add_to_history("="*60)
        add_to_history(f"🎫 Ticket: {ticket_no}")
        add_to_history(f"🏢 Department: {department}")
        add_to_history(f"👤 Resolved by: {resolved_by_text}")
        add_to_history(f"📝 Resolution: {resolution_note}")
        add_to_history(f"📊 Status updated: OPEN → CLOSED")
        add_to_history(f"⏰ Closed at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

        # Update memory
        ticket_status[ticket_no] = f"✅ Resolved by {department} Department"

        # Update data store
        for item in data_store:
            if item.get("ticket_no") == ticket_no:
                item["status"] = "CLOSED"
                break

        # Send closure email
        add_to_history(f"📧 Sending closure email to: {email}")
        email_sent = send_email(
            f"✅ Ticket {ticket_no} Resolved - {department}",
            f"""TICKET RESOLUTION NOTIFICATION

Dear {created_by},

Your complaint has been successfully resolved by the {department} Department!

Ticket Number: {ticket_no}
Department: {department}
Complaint: {complaint}
Location: {location}

Status: CLOSED
Resolved By: {resolved_by_text}
Resolution: {resolution_note}
Closed At: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Thank you for using GuardianX AI!

Best regards,
GuardianX Team
""",
            email
        )

        if email_sent:
            add_to_history(f"✅ Closure email sent successfully to {email}")
        else:
            add_to_history(f"⚠️ Email sending failed")

        add_to_history("="*60)

        return f"✅ SUCCESS!\n\nTicket {ticket_no} marked as RESOLVED\n📧 Closure email sent to: {email}\n📊 Database updated: OPEN → CLOSED\n\nYou can now close this form.", get_history_text()

    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        print(f"Resolution error: {e}")
        import traceback
        traceback.print_exc()
        add_to_history(f"❌ Error resolving ticket {ticket_no}: {str(e)}")
        return error_msg, get_history_text()

def refresh_department_view(department):
    """Refresh department view"""
    stats = get_department_stats(department)
    open_tickets = get_department_tickets_html(department)
    closed_tickets = get_department_closed_tickets_html(department)
    return stats, open_tickets, closed_tickets, get_history_text()


# ==================== ADMIN FUNCTIONS ====================
def get_ticket_stats_html():
    try:
        conn = sqlite3.connect("complaints.db")
        cur = conn.cursor()
        cur.execute("SELECT COUNT(*) FROM tickets")
        total = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM tickets WHERE status='OPEN'")
        open_count = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM tickets WHERE status='CLOSED'")
        closed_count = cur.fetchone()[0]
        conn.close()
        return _stat_row(total, open_count, closed_count)
    except Exception as e:
        return _empty_state("⚠️", "Stats unavailable", str(e))

def view_db_html():
    """Render every ticket as a compact, scannable table for the admin panel."""
    try:
        conn = sqlite3.connect("complaints.db")
        cur = conn.cursor()
        cur.execute("""
            SELECT ticket_no, created_by, department, complaint,
                   location, priority, status, created_at, closed_at
            FROM tickets ORDER BY created_at DESC
        """)
        rows = cur.fetchall()
        conn.close()

        if not rows:
            return _empty_state("🗄️", "No tickets in the database", "Tickets created by citizens will show up here.")

        body_rows = []
        for ticket_no, created_by, department, complaint, location, priority, status, created_at, closed_at in rows:
            short_complaint = str(complaint)
            if len(short_complaint) > 60:
                short_complaint = short_complaint[:60] + "…"
            body_rows.append(f"""
            <tr>
                <td class="mono">{_esc(ticket_no)}</td>
                <td>{_esc(created_by)}</td>
                <td><span class="badge badge-dept">{_esc(department)}</span></td>
                <td class="col-wide">{_esc(short_complaint)}</td>
                <td>{_esc(location)}</td>
                <td>{_priority_badge(priority)}</td>
                <td>{_status_badge(status)}</td>
                <td>{_esc(_fmt_dt(created_at))}</td>
                <td>{_esc(_fmt_dt(closed_at)) if closed_at else "—"}</td>
            </tr>
            """)

        return f"""
        <div class="gx-table-wrap">
        <table class="gx-table">
            <thead>
                <tr>
                    <th>Ticket ID</th><th>Citizen</th><th>Department</th><th>Complaint</th>
                    <th>Location</th><th>Priority</th><th>Status</th><th>Created</th><th>Closed</th>
                </tr>
            </thead>
            <tbody>{"".join(body_rows)}</tbody>
        </table>
        </div>
        """
    except Exception as e:
        print(f"❌ Database view error: {e}")
        return _empty_state("⚠️", "Couldn't load the database", str(e))


# ==================== MAIN GRADIO INTERFACE ====================
init_db()

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

:root {
    --gx-bg: #f4f6f9;
    --gx-surface: #ffffff;
    --gx-border: #e2e6ec;
    --gx-text: #1a2233;
    --gx-text-muted: #64748b;
    --gx-primary: #1d4ed8;
    --gx-primary-dark: #1e3a8a;
    --gx-primary-light: #eef2ff;
    --gx-accent: #0f766e;
    --gx-danger: #dc2626;
    --gx-success: #059669;
    --gx-warning: #d97706;
    --gx-radius: 12px;
    --gx-radius-sm: 8px;
    --gx-radius-lg: 16px;
    --gx-shadow: 0 1px 2px rgba(16,24,40,0.04), 0 1px 3px rgba(16,24,40,0.08);
    --gx-shadow-md: 0 4px 10px rgba(16,24,40,0.06), 0 2px 4px rgba(16,24,40,0.06);
}

/* Robust font stack: if Google Fonts fails to load everything must still
   fall back to a clean system sans-serif instead of the browser default. */
* { font-family: 'Inter', -apple-system, 'Segoe UI', Roboto, Arial, sans-serif !important; }
code, pre, .log-box textarea, .log-box textarea *, .mono { font-family: 'JetBrains Mono', 'Fira Code', ui-monospace, Consolas, 'Courier New', monospace !important; }

/* Force a light theme no matter the OS/browser color-scheme preference. */
html, body, .gradio-container, .dark {
    color-scheme: light !important;
    --body-background-fill: var(--gx-bg) !important;
    --body-text-color: var(--gx-text) !important;
    --background-fill-primary: #ffffff !important;
    --background-fill-secondary: #f8fafc !important;
    --block-background-fill: #ffffff !important;
    --block-border-color: var(--gx-border) !important;
    --border-color-primary: var(--gx-border) !important;
    --block-label-background-fill: transparent !important;
    --block-label-text-color: var(--gx-text-muted) !important;
    --block-title-text-color: var(--gx-text) !important;
    --input-background-fill: #ffffff !important;
    --input-border-color: #cbd5e1 !important;
    --body-text-color-subdued: var(--gx-text-muted) !important;
    --panel-background-fill: #ffffff !important;
    --form-gap-width: 0px !important;
}

/* Pure layout wrappers are marked with this class and stripped of all
   Block chrome -- this is what stops them rendering as extra empty
   white boxes anywhere in the app. */
.gx-plain, .gx-plain > .form, .gx-plain > .block {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    padding: 0 !important;
    margin: 0 !important;
    min-height: 0 !important;
}

/* Defensive CSS fallback -- kept as a first line of defense, but Gradio
   wraps every Group/Column in its own inner divs (labels, form wrappers,
   padding elements) so these containers are never truly ":empty" in the
   DOM even when they show no real content. The JS cleaner registered via
   app.load() below is what actually removes stray empty boxes app-wide,
   including the ones that appear right after switching roles/tabs. */
.card-node:empty,
.gx-plain:empty,
.gx-section:empty,
.form:empty,
.gap:empty {
    display: none !important;
}
.card-node.gx-is-empty {
    display: none !important;
}

* { box-sizing: border-box; }

.pending, .generating,
.pending *, .generating * {
    opacity: 1 !important;
    filter: none !important;
    animation: none !important;
    -webkit-text-fill-color: unset !important;
}

.block, .form, .gap, .prose, label, .wrap {
    background: transparent !important;
}

/* Gradio wraps EVERY component (Textbox, Dropdown, HTML, Dataframe, Image...)
   in its own default ".block" card -- border, rounded corners, shadow --
   even when that component already sits inside one of our ".card-node"
   cards. The result was a card-inside-a-card look, and for components
   whose content is short or hasn't rendered yet (an HTML block, an empty
   Image placeholder) that inner card reads as a mysterious empty box.
   Strip the default chrome from every block, then explicitly restore it
   only for our own ".card-node" cards further down this stylesheet. */
.gradio-container .block {
    border: none !important;
    box-shadow: none !important;
    background: transparent !important;
}

body, .gradio-container {
    background: var(--gx-bg) !important;
    color: var(--gx-text) !important;
}

.prose, .prose p, .prose li, .prose span, p, li, span, h1, h2, h3, h4, h5, h6 {
    color: var(--gx-text);
}

.gradio-container {
    max-width: 1180px !important;
    margin: 0 auto !important;
    padding-top: 20px !important;
}

/* ---------------- Header ---------------- */
.gx-header {
    display: flex;
    align-items: center;
    justify-content: space-between;
    gap: 16px;
    flex-wrap: wrap;
    padding: 22px 28px;
    background: linear-gradient(135deg, var(--gx-primary-dark), #16234f);
    border-radius: var(--gx-radius);
    margin-bottom: 20px;
}
.gx-brand { display: flex; align-items: center; gap: 14px; }
.gx-logo-mark {
    width: 46px; height: 46px; flex-shrink: 0;
    background: rgba(255,255,255,0.12);
    border: 1px solid rgba(255,255,255,0.2);
    border-radius: 10px;
    display: flex; align-items: center; justify-content: center;
}
.gx-logo-mark svg { width: 24px; height: 24px; display: block; }
.gx-brand-text { display: flex; flex-direction: column; justify-content: center; gap: 3px; }
.gx-brand-text h1 {
    margin: 0; font-size: 1.45em; line-height: 1.15;
    color: #ffffff; font-weight: 700; letter-spacing: -0.01em;
}
.gx-brand-text p {
    margin: 0; font-size: 0.86em; line-height: 1.2; color: #b7c3ea; font-weight: 400;
}
.gx-badge {
    display: flex; align-items: center; gap: 7px;
    background: rgba(255,255,255,0.08);
    border: 1px solid rgba(255,255,255,0.18);
    border-radius: 999px;
    padding: 7px 16px;
    font-size: 0.78em;
    color: #e0e7ff;
    white-space: nowrap;
}
.gx-badge::before { content: ''; width: 7px; height: 7px; border-radius: 50%; background: #34d399; flex-shrink: 0; }

/* ---------------- Session / top bar ---------------- */
.gx-topbar {
    display: flex !important;
    align-items: center !important;
    justify-content: space-between !important;
    background: var(--gx-surface) !important;
    border: 1px solid var(--gx-border) !important;
    border-radius: var(--gx-radius-sm) !important;
    padding: 12px 20px !important;
    margin-bottom: 20px !important;
    box-shadow: var(--gx-shadow);
}
.gx-topbar > * { margin: 0 !important; }
.gx-topbar p {
    font-size: 0.95em !important;
    color: var(--gx-text) !important;
    font-weight: 600 !important;
}

/* ---------------- Role sections ---------------- */
.gx-section { padding-top: 2px; }
.gx-section-title {
    font-size: 1.3em !important;
    font-weight: 700 !important;
    color: var(--gx-text) !important;
    margin: 4px 0 16px 0 !important;
}

/* ---------------- Login screen ---------------- */
.gx-login-shell { max-width: 480px !important; margin: 28px auto 0 auto !important; }
.gx-login-shell .card-node { padding: 0 !important; overflow: hidden !important; }
.gx-login-head {
    padding: 26px 28px 20px 28px;
    background: linear-gradient(135deg, var(--gx-primary-light), #ffffff);
    border-bottom: 1px solid var(--gx-border);
}
.gx-login-icon {
    width: 42px; height: 42px; border-radius: 10px;
    background: var(--gx-primary);
    display: flex; align-items: center; justify-content: center;
    margin-bottom: 12px;
}
.gx-login-icon svg { width: 22px; height: 22px; }
.gx-login-head h2 {
    margin: 0 0 4px 0; font-size: 1.2em; font-weight: 700; color: var(--gx-text);
}
.gx-login-head p {
    margin: 0; font-size: 0.88em; color: var(--gx-text-muted);
}
.gx-login-body { padding: 22px 28px 28px 28px; }

/* Login role tabs (Citizen / Department / Admin) -- rendered as a proper
   segmented control: a soft pill track with a solid white "active" pill,
   rather than the old underline-only tab strip. Keyed off role="tab" /
   aria-selected, which are stable ARIA attributes across Gradio versions. */
.gx-role-tabs [role="tablist"] {
    display: flex !important;
    flex-wrap: nowrap !important;
    gap: 4px !important;
    margin-bottom: 22px !important;
    background: #eef1f6 !important;
    border: none !important;
    border-bottom: none !important;
    border-radius: 999px !important;
    padding: 4px !important;
    position: relative !important;
}
/* Gradio's own theme draws a sliding amber/orange underline bar as a
   sibling element inside the tablist to mark the active tab. We render
   selection as a solid pill on the tab button itself instead, so that
   leftover indicator bar needs to be hidden -- otherwise it floats
   underneath our pill as a stray orange line. */
.gx-role-tabs [role="tablist"] > *:not([role="tab"]) {
    display: none !important;
}
.gx-role-tabs [role="tab"] {
    flex: 1 1 0 !important;
    min-width: 0 !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    gap: 6px !important;
    text-align: center !important;
    font-size: 0.9em !important;
    font-weight: 600 !important;
    padding: 10px 8px !important;
    white-space: nowrap !important;
    background: transparent !important;
    color: #475569 !important;
    border: none !important;
    border-bottom: none !important;
    border-radius: 999px !important;
    box-shadow: none !important;
    transition: background 0.15s ease, color 0.15s ease, box-shadow 0.15s ease !important;
}
.gx-role-tabs [role="tab"]:hover {
    color: var(--gx-primary) !important;
    background: rgba(29, 78, 216, 0.08) !important;
}
/* Selected role = a solid, unmistakably "pressed" dark pill, not just a
   color change -- this is the button people are actively on. */
.gx-role-tabs [role="tab"][aria-selected="true"],
.gx-role-tabs [role="tab"][aria-selected="true"]:hover {
    color: #ffffff !important;
    background: var(--gx-primary) !important;
    box-shadow: 0 2px 6px rgba(29, 78, 216, 0.35) !important;
}
.gx-role-tabs .icon-button-wrapper,
.gx-role-tabs button.overflow-menu {
    display: none !important;
}

/* ---------------- Cards ---------------- */
.card-node {
    background: var(--gx-surface) !important;
    border: 1px solid var(--gx-border) !important;
    border-radius: var(--gx-radius) !important;
    padding: 22px !important;
    box-shadow: var(--gx-shadow);
    margin-bottom: 18px !important;
}
.card-node h2, .card-node h3 {
    color: var(--gx-text) !important;
    font-weight: 600 !important;
    margin-top: 0 !important;
}
.card-node h2 {
    font-size: 1.08em !important;
    margin-bottom: 16px !important;
    padding-bottom: 12px !important;
    border-bottom: 1px solid var(--gx-border) !important;
}
.card-node h3 { font-size: 0.92em !important; margin-bottom: 12px !important; color: var(--gx-text-muted) !important; text-transform: uppercase; letter-spacing: 0.03em; font-weight: 700 !important; }

textarea, input, select {
    background: var(--gx-surface) !important;
    color: var(--gx-text) !important;
    border: 1px solid #cbd5e1 !important;
    border-radius: var(--gx-radius-sm) !important;
    padding: 9px 12px !important;
}

textarea:focus, input:focus, select:focus {
    border-color: var(--gx-primary) !important;
    box-shadow: 0 0 0 3px rgba(29, 78, 216, 0.12) !important;
    outline: none !important;
}

select {
    appearance: none !important;
    -webkit-appearance: none !important;
    -moz-appearance: none !important;
    padding-right: 38px !important;
    background-image: url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 20 20' fill='none' stroke='%2364748b' stroke-width='2'><path d='M5 7.5L10 12.5L15 7.5' stroke-linecap='round' stroke-linejoin='round'/></svg>") !important;
    background-repeat: no-repeat !important;
    background-position: right 12px center !important;
    background-size: 16px !important;
    cursor: pointer !important;
}

label span {
    color: var(--gx-text-muted) !important;
    font-weight: 500 !important;
    font-size: 0.85em !important;
}

input:disabled, textarea:disabled, select:disabled {
    background: #f8fafc !important;
    color: var(--gx-text) !important;
    -webkit-text-fill-color: var(--gx-text) !important;
    opacity: 1 !important;
    cursor: default !important;
}

.gx-locked-field {
    position: relative !important;
    border-left: 3px solid var(--gx-primary) !important;
    border-radius: var(--gx-radius-sm) !important;
    padding-left: 2px !important;
}
.gx-locked-field select, .gx-locked-field select:disabled {
    background-color: #f4f6fb !important;
    color: var(--gx-text) !important;
    -webkit-text-fill-color: var(--gx-text) !important;
    cursor: not-allowed !important;
    font-weight: 600 !important;
    padding-left: 38px !important;
    border: 1px solid var(--gx-border) !important;
    background-image: url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24' fill='none' stroke='%2364748b' stroke-width='2'><rect x='5' y='11' width='14' height='9' rx='2'/><path d='M8 11V7a4 4 0 0 1 8 0v4' stroke-linecap='round'/></svg>"), url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 20 20' fill='none' stroke='%2364748b' stroke-width='2'><path d='M5 7.5L10 12.5L15 7.5' stroke-linecap='round' stroke-linejoin='round'/></svg>") !important;
    background-repeat: no-repeat, no-repeat !important;
    background-position: 12px center, right 12px center !important;
    background-size: 15px, 16px !important;
}

.log-box textarea,
.log-box textarea:disabled {
    background: #0f172a !important;
    color: #86efac !important;
    -webkit-text-fill-color: #86efac !important;
    opacity: 1 !important;
    font-family: 'JetBrains Mono', 'Fira Code', ui-monospace, Consolas, 'Courier New', monospace !important;
    font-size: 12.5px !important;
    line-height: 1.6 !important;
    border: 1px solid #1e293b !important;
    border-radius: var(--gx-radius-sm) !important;
}
.log-box textarea::placeholder { color: #475569 !important; -webkit-text-fill-color: #475569 !important; }

/* ---------------- Buttons ---------------- */
button {
    background: var(--gx-primary) !important;
    color: #ffffff !important;
    font-weight: 600 !important;
    border: none !important;
    border-radius: var(--gx-radius-sm) !important;
    padding: 11px 22px !important;
    font-size: 0.92em !important;
    transition: background 0.15s ease, box-shadow 0.15s ease, transform 0.05s ease !important;
    box-shadow: none !important;
}

button:hover {
    background: var(--gx-primary-dark) !important;
    box-shadow: 0 2px 6px rgba(29, 78, 216, 0.25) !important;
    transform: none !important;
}
button:active { transform: translateY(1px) !important; }

.gx-btn-secondary button {
    background: var(--gx-surface) !important;
    color: var(--gx-primary) !important;
    border: 1px solid var(--gx-border) !important;
    box-shadow: none !important;
}
.gx-btn-secondary button:hover {
    background: var(--gx-primary-light) !important;
    border-color: var(--gx-primary) !important;
}

.gx-logout { flex: 0 0 auto !important; max-width: 130px !important; }
.gx-logout button {
    background: var(--gx-surface) !important;
    color: var(--gx-danger) !important;
    border: 1px solid #fecaca !important;
    padding: 8px 18px !important;
    font-size: 0.88em !important;
    width: auto !important;
}
.gx-logout button:hover { background: #fef2f2 !important; box-shadow: none !important; }

/* ---------------- Stat cards (dashboard summaries) ---------------- */
.gx-stat-row { display: flex; gap: 14px; flex-wrap: wrap; }
.gx-stat-card {
    flex: 1 1 140px;
    display: flex; align-items: center; gap: 12px;
    background: #f8fafc;
    border: 1px solid var(--gx-border);
    border-radius: var(--gx-radius-sm);
    padding: 14px 16px;
}
.gx-stat-icon {
    width: 38px; height: 38px; border-radius: 10px;
    display: flex; align-items: center; justify-content: center;
    font-size: 1.1em; flex-shrink: 0;
}
.gx-stat-total { background: var(--gx-primary-light); color: var(--gx-primary); }
.gx-stat-open { background: #fef3c7; color: var(--gx-warning); }
.gx-stat-closed { background: #d1fae5; color: var(--gx-success); }
.gx-stat-num { font-size: 1.35em; font-weight: 800; color: var(--gx-text); line-height: 1.1; }
.gx-stat-label { font-size: 0.78em; color: var(--gx-text-muted); font-weight: 600; margin-top: 2px; }

/* ---------------- Badges ---------------- */
.badge {
    display: inline-flex; align-items: center; gap: 4px;
    font-size: 0.72em; font-weight: 700; letter-spacing: 0.02em;
    padding: 3px 9px; border-radius: 999px; white-space: nowrap;
    text-transform: uppercase;
}
.badge-critical { background: #fee2e2; color: #b91c1c; }
.badge-high { background: #ffedd5; color: #c2410c; }
.badge-medium { background: #fef9c3; color: #a16207; }
.badge-low { background: #dcfce7; color: #15803d; }
.badge-status-open { background: #dbeafe; color: #1d4ed8; }
.badge-status-closed { background: #e2e8f0; color: #475569; }
.badge-dept { background: var(--gx-primary-light); color: var(--gx-primary); text-transform: none; }

/* ---------------- Ticket cards (Department Open/Closed views) ---------------- */
.ticket-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(300px, 1fr)); gap: 14px; }
.ticket-card {
    background: #f8fafc;
    border: 1px solid var(--gx-border);
    border-left: 3px solid var(--gx-primary);
    border-radius: var(--gx-radius-sm);
    padding: 14px 16px;
}
.ticket-card-closed { border-left-color: var(--gx-success); opacity: 0.92; }
.ticket-card-head { display: flex; align-items: center; justify-content: space-between; gap: 8px; margin-bottom: 10px; }
.ticket-id { font-family: 'JetBrains Mono', ui-monospace, Consolas, monospace; font-weight: 700; font-size: 0.85em; color: var(--gx-text); }
.ticket-badges { display: flex; gap: 6px; }
.ticket-meta { display: grid; grid-template-columns: repeat(auto-fit, minmax(120px, 1fr)); gap: 8px 14px; margin-bottom: 10px; }
.ticket-meta-item { display: flex; flex-direction: column; gap: 1px; }
.mi-label { font-size: 0.68em; font-weight: 700; text-transform: uppercase; letter-spacing: 0.04em; color: var(--gx-text-muted); }
.mi-value { font-size: 0.86em; color: var(--gx-text); font-weight: 500; }
.ticket-complaint {
    font-size: 0.86em; color: var(--gx-text); line-height: 1.5;
    background: #ffffff; border: 1px solid var(--gx-border); border-radius: 6px;
    padding: 8px 10px; margin-bottom: 4px;
}
.ticket-resolution {
    font-size: 0.83em; color: var(--gx-text); line-height: 1.5; margin-top: 8px;
    background: #ecfdf5; border: 1px solid #a7f3d0; border-radius: 6px; padding: 8px 10px;
}
.ticket-resolution .mi-label { color: #047857; }

/* ---------------- Admin "All Tickets" table ---------------- */
.gx-table-wrap { overflow-x: auto; border: 1px solid var(--gx-border); border-radius: var(--gx-radius-sm); }
.gx-table { width: 100%; border-collapse: collapse; font-size: 0.84em; }
.gx-table thead th {
    background: #f1f5f9; color: var(--gx-text-muted); text-align: left;
    font-weight: 700; text-transform: uppercase; font-size: 0.72em; letter-spacing: 0.04em;
    padding: 10px 12px; border-bottom: 1px solid var(--gx-border); white-space: nowrap;
}
.gx-table tbody td {
    padding: 10px 12px; border-bottom: 1px solid var(--gx-border);
    color: var(--gx-text); vertical-align: top; white-space: nowrap;
}
.gx-table tbody td.col-wide { white-space: normal; min-width: 220px; }
.gx-table tbody tr:last-child td { border-bottom: none; }
.gx-table tbody tr:nth-child(even) td { background: #f8fafc; }
.gx-table tbody tr:hover td { background: var(--gx-primary-light); }
.gx-table td.mono { font-family: 'JetBrains Mono', ui-monospace, Consolas, monospace; font-weight: 600; }

/* ---------------- Empty state ---------------- */
.gx-empty-state {
    display: flex; flex-direction: column; align-items: center; justify-content: center;
    text-align: center; padding: 36px 20px; color: var(--gx-text-muted);
    background: #f8fafc; border: 1px dashed var(--gx-border); border-radius: var(--gx-radius-sm);
}
.gx-empty-icon { font-size: 1.8em; margin-bottom: 8px; }
.gx-empty-title { font-weight: 700; color: var(--gx-text); font-size: 0.95em; margin-bottom: 2px; }
.gx-empty-sub { font-size: 0.82em; }

/* ---------------- Legacy table styling (kept for any raw Dataframe still in use) ---------------- */
table, .table-wrap, .dataframe, div[data-testid="dataframe"] {
    background: #ffffff !important;
    border-radius: var(--gx-radius-sm) !important;
}
.table-wrap button,
.table-wrap .icon-button-wrapper button,
button.icon-button {
    background: var(--gx-surface) !important;
    border: 1px solid var(--gx-border) !important;
    color: var(--gx-text-muted) !important;
    width: 32px !important;
    height: 32px !important;
    min-width: 32px !important;
    padding: 5px !important;
    border-radius: var(--gx-radius-sm) !important;
    box-shadow: none !important;
}
.table-wrap button:hover,
button.icon-button:hover {
    background: var(--gx-primary-light) !important;
    color: var(--gx-primary) !important;
    border-color: var(--gx-primary) !important;
}

/* ---------------- Status / helper text ---------------- */
.gx-status p { font-size: 0.86em !important; margin: 6px 0 0 0 !important; }

/* ---------------- Accordion (Analytics) ---------------- */
.gx-accordion { border: 1px solid var(--gx-border) !important; border-radius: var(--gx-radius) !important; background: var(--gx-surface) !important; box-shadow: var(--gx-shadow); margin-bottom: 18px !important; }

/* ---------------- Section divider labels above dataframes/HTML blocks ---------------- */
.gx-subhead { font-size: 0.92em !important; font-weight: 700 !important; color: var(--gx-text-muted) !important; text-transform: uppercase; letter-spacing: 0.03em; margin: 0 0 12px 0 !important; }

/* ---------------- Anti-flash reveal ----------------
   Gradio paints component "shells" before their content/values are
   filled in. On a slow connection (e.g. the public share tunnel, or
   while the model is warming up) that gap is visible as blank white
   boxes for a moment. Keep the whole app invisible until a JS timer
   (added via app.load(..., js=...) below) confirms render is settled,
   then fade it in -- so nobody ever sees the empty-shell frame. */
.gradio-container {
    opacity: 0 !important;
    transition: opacity 0.25s ease-out !important;
}
.gradio-container.gx-ready {
    opacity: 1 !important;
}
"""

# DEMO credentials only -- replace with real authentication before any production use
DEPARTMENT_PASSWORDS = {
    "Emergency": "emergency123",
    "Water": "water123",
    "Electricity": "electric123",
    "Traffic": "traffic123",
    "Garbage": "garbage123",
}
ADMIN_CREDENTIALS = {"admin": "admin@2026"}

app = gr.Blocks()

with app:
    gr.HTML("""
    <div class='gx-header'>
        <div class='gx-brand'>
            <div class='gx-logo-mark'>
                <svg viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg">
                    <path d="M12 2.5L4.5 5.5V11C4.5 16.14 7.66 20.6 12 21.9C16.34 20.6 19.5 16.14 19.5 11V5.5L12 2.5Z"
                          fill="#93c5fd" stroke="#ffffff" stroke-width="1.2" stroke-linejoin="round"/>
                    <path d="M9 12.2L11 14.2L15.2 9.8" stroke="#1e3a8a" stroke-width="1.6"
                          stroke-linecap="round" stroke-linejoin="round"/>
                </svg>
            </div>
            <div class='gx-brand-text'>
                <h1>GuardianX</h1>
                <p>Autonomous civic complaint triage &amp; dispatch platform</p>
            </div>
        </div>
        <div class='gx-badge'>DeBERTa-v3 Classifier &middot; Live</div>
    </div>
    """)

    auth_role = gr.State(value=None)

    # ==================== LOGIN SCREEN ====================
    with gr.Column(visible=True, elem_classes="gx-login-shell gx-plain") as login_screen:
        with gr.Group(elem_classes="card-node"):
            gr.HTML("""
            <div class='gx-login-head'>
                <div class='gx-login-icon'>
                    <svg viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg">
                        <path d="M12 2.5L4.5 5.5V11C4.5 16.14 7.66 20.6 12 21.9C16.34 20.6 19.5 16.14 19.5 11V5.5L12 2.5Z"
                              fill="#ffffff" stroke="#ffffff" stroke-width="1" stroke-linejoin="round"/>
                        <path d="M9 12.2L11 14.2L15.2 9.8" stroke="#1d4ed8" stroke-width="1.6"
                              stroke-linecap="round" stroke-linejoin="round"/>
                    </svg>
                </div>
                <h2>Sign In to GuardianX</h2>
                <p>Choose the portal that matches your role.</p>
            </div>
            """)
            with gr.Column(elem_classes="gx-login-body gx-plain"):
                with gr.Tabs(elem_classes="gx-role-tabs"):
                    with gr.TabItem("👤 Citizen"):
                        login_user_name = gr.Textbox(
                            label="Your Name",
                            placeholder="Enter your name"
                        )
                        login_user_email = gr.Textbox(
                            label="Email",
                            placeholder="your.email@example.com"
                        )
                        login_user_btn = gr.Button("Continue as Citizen", size="lg", variant="primary")
                        login_user_status = gr.Markdown("", elem_classes="gx-status")

                    with gr.TabItem("🏢 Department"):
                        login_dept_select = gr.Dropdown(
                            choices=["Emergency", "Water", "Electricity", "Traffic", "Garbage"],
                            value="Emergency",
                            label="Department"
                        )
                        login_dept_pass = gr.Textbox(
                            label="Department Password",
                            type="password",
                            placeholder="Enter department password"
                        )
                        login_dept_btn = gr.Button("Sign In", size="lg", variant="primary")
                        login_dept_status = gr.Markdown("", elem_classes="gx-status")

                    with gr.TabItem("🛡️ Admin"):
                        login_admin_user = gr.Textbox(
                            label="Admin Username",
                            placeholder="Enter admin username"
                        )
                        login_admin_pass = gr.Textbox(
                            label="Admin Password",
                            type="password",
                            placeholder="Enter admin password"
                        )
                        login_admin_btn = gr.Button("Sign In", size="lg", variant="primary")
                        login_admin_status = gr.Markdown("", elem_classes="gx-status")

    # ==================== MAIN APPLICATION (locked until sign-in) ====================
    with gr.Column(visible=False, elem_classes="gx-plain") as main_app:
        with gr.Row(elem_classes="gx-topbar", equal_height=True):
            session_label = gr.Markdown("", scale=8)
            logout_btn = gr.Button("Log Out", size="sm", scale=1, min_width=110, elem_classes="gx-logout")

        with gr.Column(elem_classes="gx-plain") as main_sections:  # role-specific sections; only one is made visible at a time
            # ==================== COMMAND CENTER (CITIZEN) ====================
            with gr.Column(visible=False, elem_classes="gx-section gx-plain") as tab_command:
                gr.Markdown("# Command Center", elem_classes="gx-section-title")
                with gr.Row(equal_height=True):
                    with gr.Column(scale=1, elem_classes="gx-plain"):
                        with gr.Group(elem_classes="card-node"):
                            gr.Markdown("## Citizen Input")
                            creator_inp = gr.Textbox(
                                label="Your Name",
                                placeholder="Enter your name"
                            )
                            email_inp = gr.Textbox(
                                label="Email (for updates)",
                                placeholder="your.email@example.com"
                            )
                            inp = gr.Textbox(
                                lines=8,
                                label="Complaint Description",
                                placeholder="Describe your complaint in detail..."
                            )
                            location = gr.Textbox(
                                label="Location",
                                placeholder="Enter the location...",
                                lines=2
                            )
                            submit = gr.Button("Submit Complaint", size="lg", variant="primary")

                    with gr.Column(scale=1, elem_classes="gx-plain"):
                        with gr.Group(elem_classes="card-node"):
                            gr.Markdown("## Agent Decision Panel")
                            with gr.Row():
                                agent_out = gr.Textbox(label="Primary Agent", interactive=False, scale=2)
                                sla_out = gr.Textbox(label="SLA", interactive=False, scale=1)
                            priority_out = gr.Textbox(label="Priority Level", interactive=False)
                            subs_out = gr.Textbox(label="Sub Agents", interactive=False, lines=2)
                            actions_out = gr.Textbox(
                                label="Agent Actions",
                                lines=8,
                                interactive=False
                            )

                with gr.Group(elem_classes="card-node"):
                    gr.Markdown("## System Memory")
                    hist = gr.Textbox(
                        lines=12,
                        elem_classes="log-box",
                        label="Activity Log",
                        interactive=False
                    )

            # ==================== DEPARTMENT DASHBOARD ====================
            with gr.Column(visible=False, elem_classes="gx-section gx-plain") as tab_dept:
                gr.Markdown("# Department Dashboard", elem_classes="gx-section-title")

                with gr.Group(elem_classes="card-node"):
                    with gr.Row(equal_height=True):
                        dept_selector = gr.Dropdown(
                            choices=["Emergency", "Water", "Electricity", "Traffic", "Garbage"],
                            value="Emergency",
                            label="Your Department",
                            interactive=False,
                            elem_classes="gx-locked-field",
                            scale=2
                        )
                        dept_refresh_btn = gr.Button("↻ Refresh", size="lg", elem_classes="gx-btn-secondary", scale=1)

                    dept_stats = gr.HTML(value=get_department_stats("Emergency"))

                with gr.Group(elem_classes="card-node"):
                    gr.Markdown("### Open Tickets — Need Resolution")
                    dept_open_tickets = gr.HTML(value=get_department_tickets_html("Emergency"))

                with gr.Group(elem_classes="card-node"):
                    gr.Markdown("### Resolve a Ticket")
                    with gr.Row(equal_height=True):
                        ticket_no_resolve = gr.Textbox(
                            label="Ticket Number",
                            placeholder="e.g., TKT-123456",
                            scale=1
                        )
                        resolution_note = gr.Textbox(
                            label="Resolution Details",
                            placeholder="Enter how the issue was resolved...",
                            lines=1,
                            scale=2
                        )
                    resolve_btn = gr.Button("✓ Mark Resolved", size="lg", variant="primary")
                    resolution_status = gr.Textbox(
                        label="Resolution Status",
                        interactive=False,
                        lines=3
                    )

                with gr.Group(elem_classes="card-node"):
                    gr.Markdown("### Closed Tickets — Successfully Resolved")
                    dept_closed_tickets = gr.HTML(value=get_department_closed_tickets_html("Emergency"))

                with gr.Accordion("Activity Log", open=False, elem_classes="gx-accordion"):
                    dept_hist = gr.Textbox(
                        lines=10,
                        elem_classes="log-box",
                        label="",
                        show_label=False,
                        interactive=False
                    )

            # ==================== ADMIN PANEL ====================
            with gr.Column(visible=False, elem_classes="gx-section gx-plain") as tab_admin:
                gr.Markdown("# Admin Panel", elem_classes="gx-section-title")

                with gr.Group(elem_classes="card-node"):
                    gr.Markdown("## Database Overview")
                    stats_display = gr.HTML(value=get_ticket_stats_html())
                    with gr.Row():
                        view_btn = gr.Button("↻ Refresh Database", size="lg", elem_classes="gx-btn-secondary")
                        export_btn = gr.Button("⬇ Export Records (CSV)", size="lg", elem_classes="gx-btn-secondary")
                    file_out = gr.File(label="Download CSV Export", visible=True)

                with gr.Group(elem_classes="card-node"):
                    gr.Markdown("### All Tickets")
                    db_table = gr.HTML(value=view_db_html())

                with gr.Accordion("📈 Analytics", open=False, elem_classes="gx-accordion"):
                    chart_btn = gr.Button("Generate Analytics", size="lg", elem_classes="gx-btn-secondary")
                    with gr.Row():
                        with gr.Column(scale=1, elem_classes="gx-plain"):
                            gr.Markdown("### By Agent")
                            c1 = gr.Image(height=300, show_label=False, container=False)
                        with gr.Column(scale=1, elem_classes="gx-plain"):
                            gr.Markdown("### By Day")
                            c2 = gr.Image(height=300, show_label=False, container=False)
                        with gr.Column(scale=1, elem_classes="gx-plain"):
                            gr.Markdown("### By Priority")
                            c3 = gr.Image(height=300, show_label=False, container=False)

    # ============================================================================
    # EVENT HANDLERS
    # ============================================================================

    def run_agent_wrapper(text, location, creator_name, user_email):
        result = run_agent(text, location, creator_name, user_email)
        if len(result) == 6:
            return result[0], result[1], result[2], result[3], result[4], result[5]
        else:
            return result

    submit.click(
        run_agent_wrapper,
        [inp, location, creator_inp, email_inp],
        [agent_out, subs_out, sla_out, priority_out, actions_out, hist]
    )

    auto_refresh = gr.Timer(value=4, active=True)
    auto_refresh.tick(
        fn=lambda: get_history_text(),
        inputs=None,
        outputs=[hist, dept_hist],
        show_progress="hidden"
    )

    dept_selector.change(
        refresh_department_view,
        inputs=[dept_selector],
        outputs=[dept_stats, dept_open_tickets, dept_closed_tickets, dept_hist]
    )

    dept_refresh_btn.click(
        refresh_department_view,
        inputs=[dept_selector],
        outputs=[dept_stats, dept_open_tickets, dept_closed_tickets, dept_hist]
    )

    def handle_resolve(ticket_no, resolution_note, department):
        status_msg, history = mark_ticket_resolved(ticket_no, resolution_note, department)
        stats, open_tickets, closed_tickets, updated_history = refresh_department_view(department)
        return status_msg, updated_history, stats, open_tickets, closed_tickets, "", ""

    resolve_btn.click(
        handle_resolve,
        inputs=[ticket_no_resolve, resolution_note, dept_selector],
        outputs=[resolution_status, dept_hist, dept_stats, dept_open_tickets, dept_closed_tickets, ticket_no_resolve, resolution_note]
    )

    chart_btn.click(generate_charts, None, [c1, c2, c3])
    export_btn.click(export_admin, None, file_out)

    def refresh_admin():
        return get_ticket_stats_html(), view_db_html()

    view_btn.click(refresh_admin, None, [stats_display, db_table])

    # ==================== AUTH HANDLERS ====================
    def login_as_user(name, email):
        if not name or not name.strip():
            return (
                gr.update(), gr.update(),
                gr.update(), gr.update(), gr.update(),
                gr.update(), gr.update(),
                None, "Please enter your name to continue."
            )
        return (
            gr.update(visible=False),              # login_screen
            gr.update(visible=True),                # main_app
            gr.update(visible=True),                # tab_command
            gr.update(visible=False),               # tab_dept
            gr.update(visible=False),               # tab_admin
            f"**Signed in as Citizen** \u00b7 {name}",  # session_label
            name,                                    # creator_inp
            email,                                    # email_inp
            "user",                                    # auth_role
            ""                                          # login_user_status
        )

    login_user_btn.click(
        login_as_user,
        inputs=[login_user_name, login_user_email],
        outputs=[login_screen, main_app, tab_command, tab_dept, tab_admin,
                 session_label, creator_inp, email_inp, auth_role, login_user_status]
    )

    def login_as_department(department, password):
        if DEPARTMENT_PASSWORDS.get(department) != password:
            return (
                gr.update(), gr.update(),
                gr.update(), gr.update(), gr.update(),
                gr.update(), gr.update(),
                gr.update(), gr.update(), gr.update(), gr.update(),
                None, "Incorrect department password."
            )
        stats, open_tickets, closed_tickets, dept_history = refresh_department_view(department)
        return (
            gr.update(visible=False),                     # login_screen
            gr.update(visible=True),                       # main_app
            gr.update(visible=False),                      # tab_command
            gr.update(visible=True),                       # tab_dept
            gr.update(visible=False),                      # tab_admin
            f"**Signed in as {department} Department**",   # session_label
            department,                                     # dept_selector
            stats, open_tickets, closed_tickets, dept_history,
            department,                                       # auth_role
            ""                                                  # login_dept_status
        )

    login_dept_btn.click(
        login_as_department,
        inputs=[login_dept_select, login_dept_pass],
        outputs=[login_screen, main_app, tab_command, tab_dept, tab_admin,
                 session_label, dept_selector, dept_stats, dept_open_tickets, dept_closed_tickets, dept_hist,
                 auth_role, login_dept_status]
    )

    def login_as_admin(username, password):
        if ADMIN_CREDENTIALS.get(username) != password:
            return (
                gr.update(), gr.update(),
                gr.update(), gr.update(), gr.update(),
                gr.update(), None, "Incorrect admin credentials."
            )
        return (
            gr.update(visible=False),                # login_screen
            gr.update(visible=True),                  # main_app
            gr.update(visible=False),                 # tab_command
            gr.update(visible=False),                 # tab_dept
            gr.update(visible=True),                  # tab_admin
            f"**Signed in as Admin** \u00b7 {username}",  # session_label
            "admin",                                    # auth_role
            ""                                            # login_admin_status
        )

    login_admin_btn.click(
        login_as_admin,
        inputs=[login_admin_user, login_admin_pass],
        outputs=[login_screen, main_app, tab_command, tab_dept, tab_admin,
                 session_label, auth_role, login_admin_status]
    )

    def logout():
        return (
            gr.update(visible=True),   # login_screen
            gr.update(visible=False),  # main_app
            "",                         # session_label
            None,                        # auth_role
            "", "",                       # login_user_name, login_user_email
            "",                             # login_dept_pass
            "", "",                          # login_admin_user, login_admin_pass
            "", "", ""                        # status messages
        )

    logout_btn.click(
        logout,
        inputs=None,
        outputs=[login_screen, main_app, session_label, auth_role,
                 login_user_name, login_user_email, login_dept_pass,
                 login_admin_user, login_admin_pass,
                 login_user_status, login_dept_status, login_admin_status]
    )

    # Reveal the app only once Gradio has actually finished rendering,
    # instead of the instant the page shell appears -- this is what
    # prevents the "empty white box" flash on slower connections.
    app.load(
        fn=None,
        inputs=None,
        outputs=None,
        js="""
        () => {
            const reveal = () => document.querySelector('.gradio-container')?.classList.add('gx-ready');
            // small buffer so late-filling HTML/stat components are already in
            setTimeout(reveal, 900);

            // ---- Empty card-node cleaner ----
            // Gradio paints each Group's wrapper div immediately, then streams
            // in its Markdown/HTML/Textbox children a moment later. Right after
            // login (or a role/tab switch) several of those children haven't
            // filled in yet, so their card-node wrapper briefly -- and
            // sometimes persistently -- shows as an empty white box. A plain
            // CSS ":empty" rule can't catch this because the wrapper always
            // contains Gradio's own inner form/block divs, even when nothing
            // meaningful has rendered inside them. This walks every card and
            // hides any that have no visible text and no real content
            // (input, button, image, table, etc.), then re-checks on every
            // DOM change so cards reappear the instant their content lands.
            const gxCleanEmptyCards = () => {
                document.querySelectorAll('.card-node').forEach((card) => {
                    // Never touch the login screen. It's not where the empty-
                    // box bug lives, and it's the one screen a user can never
                    // recover from if this heuristic ever gets it wrong --
                    // so it's out of scope for this cleaner, unconditionally.
                    if (card.closest('.gx-login-shell')) return;

                    // Our own "gx-is-empty" class (added below) also sets
                    // display:none, which makes offsetParent null for the
                    // exact same reason a genuinely app-hidden ancestor
                    // would -- so a card we hid on a previous pass could
                    // never be re-examined or un-hidden again. Strip our own
                    // marker first so the check below can tell "a real
                    // ancestor is hiding this" apart from "we hid this
                    // ourselves last time and it might have content now".
                    const wasMarkedEmpty = card.classList.contains('gx-is-empty');
                    if (wasMarkedEmpty) card.classList.remove('gx-is-empty');

                    // Skip cards inside an already-hidden tab/section --
                    // nothing to clean up, and touching them can fight the
                    // show/hide logic that's hiding them on purpose.
                    if (card.offsetParent === null) {
                        if (wasMarkedEmpty) card.classList.add('gx-is-empty');
                        return;
                    }

                    const hasText = (card.innerText || '').trim().length > 0;
                    // A bare, label-less, value-less input/textarea/select is
                    // exactly the kind of stray empty pill we want gone, so it
                    // is deliberately NOT treated as "real content" here --
                    // only genuinely meaningful elements count (an input WITH
                    // a label still passes via hasText, since the label text
                    // is part of the card's innerText).
                    //
                    // Existence alone isn't enough, though: when a whole
                    // section (e.g. the Department tab) is toggled invisible,
                    // Gradio adds a "hidden" class to the individual
                    // row/button/form elements INSIDE it rather than to the
                    // Group wrapper itself -- so a <button> or <svg> tag can
                    // still be sitting in the DOM, just with zero rendered
                    // size. Checking offsetWidth/offsetHeight (not mere
                    // querySelector presence) is what tells a real, visible
                    // control apart from one of those collapsed leftovers.
                    let hasVisibleContent = false;
                    card.querySelectorAll('button, img, table, canvas, svg, video, a[href]').forEach((el) => {
                        if (el.offsetWidth > 0 || el.offsetHeight > 0) hasVisibleContent = true;
                    });

                    if (!hasText && !hasVisibleContent) {
                        card.classList.add('gx-is-empty');
                    } else {
                        card.classList.remove('gx-is-empty');
                    }
                });
            };

            gxCleanEmptyCards();
            // NOTE: this must NOT watch attribute changes (class/style).
            // gxCleanEmptyCards toggles the "gx-is-empty" class on cards as
            // part of its own work, and an attribute-watching observer would
            // see that as a new mutation and re-fire itself -- forever,
            // pegging the tab at 100% CPU until the browser calls it
            // unresponsive. childList/characterData only: real Gradio
            // content updates (text streaming in, nodes swapped) still
            // trigger a re-check; our own class toggle does not.
            let gxCleanScheduled = false;
            const gxScheduleClean = () => {
                if (gxCleanScheduled) return;
                gxCleanScheduled = true;
                requestAnimationFrame(() => {
                    gxCleanScheduled = false;
                    gxCleanEmptyCards();
                });
            };
            new MutationObserver(gxScheduleClean)
                .observe(document.body, { childList: true, subtree: true, characterData: true });
            // Safety-net poll in case a change slips past the observer
            // (e.g. a value swap that doesn't touch the DOM tree shape).
            setInterval(gxCleanEmptyCards, 500);
        }
        """
    )

    app.launch(share=True, debug=True, css=custom_css)
